In [ ]:
!git clone https://github.com/chowin131/temp.git
%cd /content/temp

In [ ]:
# 결과를 Drive에 저장
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/ell_result
!rm -rf result  # clone으로 딸려온 result 복사본 삭제 (Drive/GitHub 원본은 그대로)
!ln -s /content/drive/MyDrive/ell_result result
!ls result

In [ ]:
import torch
import main

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch  :", torch.__version__)
print("device :", device)

In [ ]:
import architectures
import datasets
import methods

print("method :", list(methods.METHOD_REGISTRY))
print("dataset:", list(datasets.NUM_CLASSES))
print()
for name, (cls, kwargs) in architectures.PRESETS.items():
    print(f"  {name:16s} {cls.__name__:18s} {kwargs}")

### FractalNet

In [ ]:
result = main.run_experiment(arch="fractalnet20")

### ViT

In [ ]:
result = main.run_experiment(
    arch="vit_ti4", lr=0.01, weight_decay=5e-5,
)

### MLP-Mixer

In [ ]:
result = main.run_experiment(
    arch="mlpmixer_8", lr=0.01, weight_decay=5e-5,
)

### ConvMixer

In [ ]:
result = main.run_experiment(arch="convmixer_256_8")

### ResNet (supervised)

In [ ]:
# RotNet과 비교하는 용도
result = main.run_experiment(arch="resnet20", method_name="supervised")

### RotNet

In [ ]:
# pretraining, knn으로 encoder 평가
result = main.run_experiment(arch="resnet20", method_name="rotnet")

In [ ]:
# fine tuning
result = main.run_experiment(
    arch="resnet20", method_name="supervised",
    encoder_weight="result/resnet20_cifar10_rotnet.pth",
)

### SimCLR

In [ ]:
# pretraining
result = main.run_experiment(
    arch="resnet20", method_name="simclr",
    batch_size=512, epochs=100, lr=0.12, weight_decay=5e-4,
    schedule="cosine", warmup_epochs=10,
    method_kwargs=dict(temperature=0.5),
)

In [ ]:
# linear evaluation, encoder freeze하고 linear만 학습
result = main.run_experiment(
    arch="resnet20", method_name="linear",
    encoder_weight="result/resnet20_cifar10_simclr.pth",
    batch_size=256, epochs=50, lr=0.1, weight_decay=0,
    schedule="cosine", knn_interval=100,
)

### MoCo v2

In [ ]:
# pretraining
result = main.run_experiment(
    arch="resnet20", method_name="moco",
    batch_size=512, epochs=100, lr=0.12, weight_decay=5e-4,
    schedule="cosine", warmup_epochs=10,
    method_kwargs=dict(queue_size=4096, momentum=0.99, temperature=0.1),
)

In [ ]:
# linear evaluation, encoder freeze하고 linear만 학습
result = main.run_experiment(
    arch="resnet20", method_name="linear",
    encoder_weight="result/resnet20_cifar10_moco.pth",
    batch_size=256, epochs=50, lr=0.1, weight_decay=0,
    schedule="cosine", knn_interval=100,
)

### ResNet18 (supervised)

In [ ]:
# 비교용
result = main.run_experiment(arch="resnet18", method_name="supervised")

### 저장 및 출력

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import utils

runs = {}
for path in sorted(glob.glob("result/*.json")):
    name = os.path.splitext(os.path.basename(path))[0]
    runs[name] = utils.load_result(path)

print(f"{len(runs)}개 run :", list(runs))

In [ ]:
# 요약 표 (pretraining run은 test acc 대신 knn acc)
def score_list(r):
    return r["test_acc"] if r["test_acc"] else r["knn_acc"]

print(f"{'run':32s} {'epochs':>7s} {'best acc':>9s} {'final acc':>10s}")
print("-" * 62)
for name, r in sorted(runs.items(), key=lambda kv: -max(score_list(kv[1]))):
    acc = score_list(r)
    tag = "" if r["test_acc"] else "  (knn)"
    print(f"{name:32s} {len(r['train_loss']):7d} "
          f"{max(acc) * 100:8.2f}% {acc[-1] * 100:9.2f}%{tag}")


In [ ]:
# knn 정확도 (학습 없이 encoder 평가)
for name, r in sorted(runs.items()):
    if r.get("knn_acc"):
        print(f"{name:36s} epoch {r['knn_epoch'][-1]:3d} | {r['knn_acc'][-1] * 100:.2f}%")

In [ ]:
# run 하나 자세히 보기 (loss / test acc / lr)
name = list(runs)[0]
utils.plot_result(runs[name], title=name)
plt.show()

In [ ]:
# 여러 run 겹쳐 그리기
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, r in sorted(runs.items()):
    epochs = range(1, len(r["train_loss"]) + 1)
    axes[0].plot(epochs, r["train_loss"], label=name)
    if r["test_acc"]:
        axes[1].plot(epochs, [a * 100 for a in r["test_acc"]], label=name)

axes[0].set_xlabel("epoch")
axes[0].set_ylabel("train loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test acc (%)")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()